# Train Nôm character detector (CenterNet) trên Kaggle

Trainer tự **đánh giá VAL F1 + count-err mỗi epoch** → biết ngay **đạt hay chưa**.

**Trước khi Run:**
1. Upload thư mục `kaggle_det_pkg/` (193MB) làm 1 Kaggle **Dataset**.
2. Notebook này: **Add Input** → chọn dataset đó.
3. **Settings → Accelerator = `GPU T4 x2`** ⚠️ (ĐỪNG chọn P100 — PyTorch mới của Kaggle không hỗ trợ P100/sm_60). **Internet = On**.
4. **Run All**. (Cell tự dò dataset, không cần sửa tên.)

In [ ]:
import os, glob, torch
ok = torch.cuda.is_available()
name = torch.cuda.get_device_name(0) if ok else 'NONE'
print('GPU:', name)
if (not ok) or ('P100' in name):
    print('⚠️  ĐỔI Accelerator sang **GPU T4 x2** (Settings -> Accelerator).')
    print('    PyTorch mới của Kaggle KHÔNG hỗ trợ P100 (sm_60); T4 (sm_75) thì chạy được.')
print('inputs:', os.listdir('/kaggle/input'))
# Tự tìm thư mục chứa detect_manifest.json (khỏi phụ thuộc tên dataset / lớp thư mục)
hits = glob.glob('/kaggle/input/**/detect_manifest.json', recursive=True)
assert hits, 'Không thấy detect_manifest.json trong /kaggle/input — đã Add đúng Dataset chưa?'
SRC = os.path.dirname(hits[0])
print('SRC =', SRC, '| files:', os.listdir(SRC)[:6])

In [ ]:
# Chuẩn bị thư mục làm việc: copy file nhỏ + symlink images (khỏi copy 193MB)
import shutil, os
os.chdir('/kaggle/working')
for f in ['train_centernet.py', 'count_constrained.py', 'detect_manifest.json']:
    shutil.copy(f'{SRC}/{f}', f)
if not os.path.exists('images'):
    os.symlink(f'{SRC}/images', 'images')
import json
man = json.load(open('detect_manifest.json'))
print('pages:', len(man), '| boxes:', sum(m['n_boxes'] for m in man), '| sample img exists:', os.path.exists(man[0]['image']))

## Train + đánh giá (≈1–2h cho 40 epoch / T4)
Mỗi epoch in `VAL F1 / P / R / count-err`. Lưu `detector.pt` (last) + `detector.best.pt` (F1 cao nhất).

**Muốn đẩy HuggingFace:** điền `HF_REPO` + `HF_TOKEN` ở cell dưới (dán token trực tiếp, hoặc dùng Kaggle Secret tên `HF_TOKEN`).

In [ ]:
# ===== ĐẨY HUGGINGFACE: điền 2 dòng dưới rồi Run (để trống = chỉ lưu Kaggle Output) =====
HF_REPO  = 'mdnt571/nom-char-det'   # repo model HF của bạn
HF_TOKEN = ''                       # dùng Kaggle Secret tên HF_TOKEN (KHUYẾN NGHỊ, đừng dán token vào đây)
# =======================================================================================
import os, subprocess, sys
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
elif not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
push = bool(HF_REPO and os.environ.get('HF_TOKEN'))
print('HF push:', f'sẽ đẩy -> {HF_REPO} (sau khi train)' if push else 'OFF (chỉ lưu Kaggle Output)')

# 1) TRAIN — img 1024 (chữ to hơn ~1.3x so 768 -> định vị box chính xác hơn -> F1 cao hơn).
#    Lần trước img=768 cho F1 0.72 (chưa thắng midpoint); 1024 là đòn rẻ nhất để nâng.
subprocess.run([sys.executable, 'train_centernet.py', '--manifest', 'detect_manifest.json',
                '--img', '1024', '--epochs', '40', '--batch', '6', '--val-frac', '0.1',
                '--out', '/kaggle/working/detector.pt'], check=True)

# 2) ĐẨY HF TỪ NOTEBOOK (sau khi train)
if push:
    from huggingface_hub import HfApi, create_repo
    tok = os.environ['HF_TOKEN']
    create_repo(HF_REPO, repo_type='model', exist_ok=True, token=tok)
    api = HfApi()
    for f in ['/kaggle/working/detector.best.pt', '/kaggle/working/detector.pt']:
        if os.path.exists(f):
            api.upload_file(path_or_fileobj=f, path_in_repo=os.path.basename(f),
                            repo_id=HF_REPO, repo_type='model', token=tok)
            print('  pushed', os.path.basename(f), '->', HF_REPO)

## Đọc kết quả — ĐẠT hay chưa?
| box F1 @IoU0.5 | median count-err | Kết luận |
|---|---|---|
| **≥ ~0.85** | **~0** | ĐẠT → tải `detector.best.pt` về, dùng `--reseg detector` |
| < 0.8 | ≥ 1 | chưa đạt → train mới (cell cuối) |

In [ ]:
# Xem chỉ số VAL đã lưu trong checkpoint tốt nhất
import torch
ck = torch.load('/kaggle/working/detector.best.pt', map_location='cpu')
print('BEST VAL:', ck.get('val'))
print('\n=== Lấy model về máy (2 cách) ===')
print('A) Kaggle Output: tab Output/Data -> tải detector.best.pt')
print('B) HuggingFace (nếu đã đặt HF_REPO): trên máy chạy ->')
print('   huggingface-cli download <HF_REPO> detector.best.pt --local-dir evaluation/ver_new/char_detector')
print('   rồi: mv evaluation/ver_new/char_detector/detector.best.pt evaluation/ver_new/char_detector/detector.pt')
print('   và: build_dataset.py --use-s3 --reseg detector --out dataset_out_det')

## (Nếu CHƯA đạt) train mới — rẻ → đắt
1. **Ảnh lớn hơn + nhiều epoch:** `--img 1024 --epochs 70`.
2. **Pretrain TKH/MTHv2** (mạnh nhất, cùng miền ván khắc): thêm dataset `HCIILAB/TKH_MTH_Datasets_Release`,
   train detector trên đó trước (tự viết manifest TKH theo cùng format), rồi `--init tkh_detector.pt`.
3. **Nhãn sạch hơn:** dùng manifest từ `bootstrap_boxes.py --complete-only` (ít box thiếu).

In [ ]:
# Ví dụ train lại ảnh lớn hơn + nhiều epoch hơn (chạy khi cần)
# !python train_centernet.py --manifest detect_manifest.json --img 1024 \
#         --epochs 70 --batch 4 --val-frac 0.1 --out /kaggle/working/detector_v2.pt